# 03 — Exportación de datos para Power BI
## easyMoney | TFM Data Science & AI — Nuclio School

**Prerequisitos:** `01-eda.ipynb` y `02-eda-deep-dive.ipynb` ejecutados

**Objetivo:** Exportar los datos en formato CSV para construir el dashboard de BI en Power BI Desktop.

**Input:** `master_df_flags.parquet` — tabla maestra con flags de calidad

**Criterio de filtrado:**
- `period_summary`, `product_penetration`, `client_profile` → **universo completo** (con anomalías incluidas como flags)
- `revenue_*`, `penetration_*`, `contracts_by_type_v2` → **solo clientes limpios** (`df_clean`, sin anomalías)
- `contracts_by_type` → **universo completo** (para comparación; usar `contracts_by_type_v2` para análisis)

**Outputs (22 CSVs):**

| # | Archivo | Descripción |
|---|---------|-------------|
| 1 | `period_summary.csv` | Evolución temporal de KPIs (17 períodos) |
| 2 | `product_penetration.csv` | Penetración por producto y período |
| 3 | `client_profile.csv` | Perfil demográfico Mayo 2019 (~443K clientes snapshot, flags incluidas) |
| 4 | `kpi_summary.csv` | KPIs calculados por período |
| 5 | `product_penetration_long.csv` | Penetración por producto (formato largo) |
| 6 | `age_sort.csv` | Orden numérico para age_group en Power BI |
| 7 | `salary_sort.csv` | Orden numérico para salary_group en Power BI |
| 8 | `contracts_by_type.csv` | Nuevas contrataciones por tipo cliente y período |
| 9 | `segment_by_period.csv` | Evolución de segmentos por período |
| 10 | `revenue_by_period.csv` | Revenue estimado (proxy) por período (17 meses) |
| 11 | `revenue_by_product.csv` | Revenue estimado (proxy) por producto, Mayo 2019 |
| 12 | `revenue_by_region.csv` | Revenue estimado (proxy) por región, Mayo 2019 |
| 13 | `product_family_map.csv` | Lookup: 14 productos → 5 familias |
| 14 | `revenue_by_family.csv` | Revenue estimado (proxy) agrupado por familia, Mayo 2019 |
| 15 | `penetration_by_family.csv` | Penetración agrupada por familia, Mayo 2019 |
| 16 | `region_map.csv` | Nombre de provincia por código |
| 17 | `contracts_by_type_v2.csv` | Contratos por tipo corregidos (new_contracts real para nuevos; excluye anomalías) |
| 18 | `crosssell_funnel.csv` | Evolución del funnel de cross-sell por período (0/1/2+/3+ productos) |
| 19 | `kpi_rentabilidad.csv` | ARPU proxy y revenue estimado por período |
| 20 | `retention_by_period.csv` | Tasa de retención y churn mensual |
| 21 | `revenue_by_segment.csv` | Revenue estimado por segmento comercial (TOP/PARTICULARES/UNIVERSITARIO) |
| 22 | `revenue_uplift.csv` | Revenue uplift por número de productos contratados |

---

In [1]:
# ── Setup: imports, paths y diccionarios de referencia ──────────────
import pandas as pd
import os

DATA_PATH   = '../../data/processed/master_df_flags.parquet'
OUTPUT_PATH = '../../data/processed/powerbi/'

df = pd.read_parquet(DATA_PATH)
os.makedirs(OUTPUT_PATH, exist_ok=True)

product_cols = ['short_term_deposit','loans','mortgage','funds','securities',
                'long_term_deposit','credit_card','payroll','pension_plan',
                'payroll_account','emc_account','debit_card','em_account_p',
                'em_acount']

last_partition = df['pk_partition'].max()

# ── Diccionario de etiquetas (usado en varias tablas) ──────────────────────
label_map = {
    'em_acount':          'Cuenta easyMoney',
    'payroll':            'Domiciliaciones',
    'em_account_p':       'Cuenta easyMoney+',
    'debit_card':         'Tarjeta débito',
    'credit_card':        'Tarjeta crédito',
    'payroll_account':    'Cuenta nómina',
    'emc_account':        'Cuenta Crypto',
    'short_term_deposit': 'Depósito C/P',
    'long_term_deposit':  'Depósito L/P',
    'pension_plan':       'Plan pensiones',
    'funds':              'Fondos inversión',
    'securities':         'Valores',
    'mortgage':           'Hipoteca',
    'loans':              'Préstamos'
}

# ── Precios unitarios estimados (usado en revenue) ─────────────────────────
price_map = {
    'em_acount':          10,
    'emc_account':        10,
    'payroll_account':    10,
    'payroll':            10,
    'debit_card':         10,
    'em_account_p':       10,
    'short_term_deposit': 40,
    'long_term_deposit':  40,
    'funds':              40,
    'securities':         40,
    'pension_plan':       40,
    'credit_card':        40,
    'loans':              60,
    'mortgage':           60
}

# ── Verificación ───────────────────────────────────────────────────────────
# ── Filtrado de anomalías ─────────────────────────────────────────────────────────────────────────
anomaly_flags = ['age_anomaly', 'deceased_anomaly', 
                'entry_date_anomaly', 'salary_anomaly']
df_clean = df[~df[anomaly_flags].any(axis=1)]

last = df_clean[df_clean['pk_partition'] == last_partition]
nc_new = last[last['is_new_client']==1]['new_contracts'].sum()
print(f"✓ Parquet cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"✓ Flags: {[c for c in df.columns if 'anomaly' in c]}")
print(f"✓ df_clean: {df_clean.shape[0]:,} filas (sin anomalías)")
print(f"✓ Última partición: {last_partition}")
print(f"✓ new_contracts clientes nuevos (May 2019): {nc_new:,}  ← debe ser > 0")
print(f"✓ label_map: {len(label_map)} productos")
print(f"✓ price_map: {len(price_map)} productos")

✓ Parquet cargado: 5,962,924 filas × 38 columnas
✓ Flags: ['age_anomaly', 'deceased_anomaly', 'entry_date_anomaly', 'salary_anomaly']
✓ df_clean: 5,940,692 filas (sin anomalías)
✓ Última partición: 2019-05-28 00:00:00
✓ new_contracts clientes nuevos (May 2019): 1,690  ← debe ser > 0
✓ label_map: 14 productos
✓ price_map: 14 productos


In [2]:
# ── Exportar CSVs para Power BI ───────────────────────────────────────────
# Nota: period_summary y product_penetration usan df completo (incluye anomalías)
# client_profile también usa df completo — las flags de anomalía se incluyen como
# columnas para que Power BI pueda filtrar según necesidad
# Para revenue y penetración (análisis cuantitativo), se usa df_clean (sin anomalías)

anomaly_flags = ['age_anomaly', 'deceased_anomaly', 
                'entry_date_anomaly', 'salary_anomaly']

# Tabla 1: Resumen por período
period_summary_export = df.groupby('pk_partition').agg(
    total_clients        = ('pk_cid', 'count'),
    active_clients       = ('active_customer', 'sum'),
    new_clients          = ('is_new_client', 'sum'),
    new_contracts        = ('new_contracts', 'sum'),
    avg_products         = ('total_products', 'mean'),
    clients_0_products   = ('total_products', lambda x: (x==0).sum()),
    clients_1_product    = ('total_products', lambda x: (x==1).sum()),
    clients_2plus        = ('total_products', lambda x: (x>=2).sum()),
).reset_index()
period_summary_export['pk_partition'] = period_summary_export['pk_partition'].astype(str).str[:10]
period_summary_export.to_csv(OUTPUT_PATH + 'period_summary.csv', index=False)
print(f"✓ period_summary.csv — {period_summary_export.shape}")

# Tabla 2: Penetración por producto por período
product_penetration = df.groupby('pk_partition')[product_cols].mean().mul(100).round(2).reset_index()
product_penetration['pk_partition'] = product_penetration['pk_partition'].astype(str).str[:10]
product_penetration.to_csv(OUTPUT_PATH + 'product_penetration.csv', index=False)
print(f"✓ product_penetration.csv — {product_penetration.shape}")

# Tabla 3: Perfil cliente última partición — SIN filtro de anomalías
# Las flags se incluyen como columnas para filtrar desde Power BI si se necesita
# Esto garantiza consistencia con period_summary (mismo universo de clientes)
client_profile = df[df['pk_partition'] == last_partition][[
    'pk_cid', 'segment', 'age_group', 'salary_group',
    'gender', 'region_code', 'country_id',
    'total_products', 'is_new_client',
    'client_age_months', 'active_customer'
] + product_cols + anomaly_flags].copy()
client_profile['pk_partition'] = str(last_partition)[:10]
client_profile.to_csv(OUTPUT_PATH + 'client_profile.csv', index=False)

n_clean    = (~client_profile[anomaly_flags].any(axis=1)).sum()
n_anomaly  = client_profile[anomaly_flags].any(axis=1).sum()
print(f"✓ client_profile.csv — {client_profile.shape}")
print(f"  → {n_clean:,} clientes sin anomalía  |  {n_anomaly:,} con anomalía (flags incluidas)")

# Definición de df_clean para tablas de revenue y análisis cuantitativo
df_clean = df[~df[anomaly_flags].any(axis=1)]

# Tabla 4: KPIs resumen
kpi_summary = period_summary_export.copy()
kpi_summary['pct_new_clients'] = (kpi_summary['new_clients'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_0_products']  = (kpi_summary['clients_0_products'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_1_product']   = (kpi_summary['clients_1_product'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_crosssell']   = (kpi_summary['clients_2plus'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary.to_csv(OUTPUT_PATH + 'kpi_summary.csv', index=False)
print(f"✓ kpi_summary.csv — {kpi_summary.shape}")

print(f"\n✓ Todos los archivos exportados en: {OUTPUT_PATH}")

✓ period_summary.csv — (17, 9)
✓ product_penetration.csv — (17, 15)
✓ client_profile.csv — (442995, 30)
  → 441,752 clientes sin anomalía  |  1,243 con anomalía (flags incluidas)
✓ kpi_summary.csv — (17, 13)

✓ Todos los archivos exportados en: ../../data/processed/powerbi/


In [3]:
# ── Tabla 5: Penetración por producto (formato largo para Power BI) ────────
# Nota: label_map definido en celda Setup
# Nota: df_clean y anomaly_flags definidos en celda anterior (Tablas 1-4)

df_last = df_clean[df_clean['pk_partition'] == last_partition]

penetration_long = pd.DataFrame({
    'producto':        list(label_map.values()),
    'nombre_tecnico':  list(label_map.keys()),
    'penetracion_pct': [df_last[col].mean() * 100 for col in label_map.keys()]
}).round(2).sort_values('penetracion_pct', ascending=False)

penetration_long.to_csv(OUTPUT_PATH + 'product_penetration_long.csv', index=False)
print(f"✅ Tabla 5: product_penetration_long.csv — {penetration_long.shape}")
print(f"\nVista previa:")
print(penetration_long.to_string(index=False))

✅ Tabla 5: product_penetration_long.csv — (14, 3)

Vista previa:
         producto     nombre_tecnico  penetracion_pct
 Cuenta easyMoney          em_acount            67.02
   Tarjeta débito         debit_card             9.78
    Cuenta nómina    payroll_account             6.00
    Cuenta Crypto        emc_account             5.59
   Plan pensiones       pension_plan             3.92
  Domiciliaciones            payroll             3.69
     Depósito L/P  long_term_deposit             1.38
  Tarjeta crédito        credit_card             1.08
          Valores         securities             0.40
 Fondos inversión              funds             0.30
        Préstamos              loans             0.01
         Hipoteca           mortgage             0.01
Cuenta easyMoney+       em_account_p             0.00
     Depósito C/P short_term_deposit             0.00


In [4]:
# ── Tablas 6-9: Tablas de referencia y evolución temporal ─────────────────

# Tabla 6: Sort order para age_group (eje ordinal en Power BI)
age_sort = pd.DataFrame({
    'age_group': ['<18', '18-24', '25-35', '35-45', '45-55', '55-65', '65+'],
    'age_sort':  [1, 2, 3, 4, 5, 6, 7]
})
age_sort.to_csv(OUTPUT_PATH + 'age_sort.csv', index=False)
print(f"✅ Tabla 6: age_sort.csv — {age_sort.shape}")

# Tabla 7: Sort order para salary_group (eje ordinal en Power BI)
salary_sort = pd.DataFrame({
    'salary_group': ['sin_ingreso', '<20k', '20-40k', '40-60k', '60-80k', '80-120k', '120k+'],
    'salary_sort':  [1, 2, 3, 4, 5, 6, 7]
})
salary_sort.to_csv(OUTPUT_PATH + 'salary_sort.csv', index=False)
print(f"✅ Tabla 7: salary_sort.csv — {salary_sort.shape}")

# Tabla 8: Nuevos contratos por tipo de cliente y período
# Nota: excluye la primera partición (sin referencia anterior para calcular new_contracts)
contracts_type = df[df['pk_partition'] != df['pk_partition'].min()].groupby(
    ['pk_partition', 'is_new_client']
).agg(
    contracts    = ('new_contracts', 'sum'),
    client_count = ('pk_cid', 'count')
).reset_index()
contracts_type['pk_partition'] = contracts_type['pk_partition'].astype(str).str[:10]
contracts_type['tipo'] = contracts_type['is_new_client'].map({0: 'Existente', 1: 'Nuevo'})
contracts_type.to_csv(OUTPUT_PATH + 'contracts_by_type.csv', index=False)
print(f"✅ Tabla 8: contracts_by_type.csv — {contracts_type.shape}")

# Tabla 9: Evolución de segmentos por período
segment_period = df.groupby(['pk_partition', 'segment'])['pk_cid'].count().reset_index()
segment_period.columns = ['pk_partition', 'segment', 'clientes']
segment_period['pk_partition'] = segment_period['pk_partition'].astype(str).str[:10]
segment_period.to_csv(OUTPUT_PATH + 'segment_by_period.csv', index=False)
print(f"✅ Tabla 9: segment_by_period.csv — {segment_period.shape}")

✅ Tabla 6: age_sort.csv — (7, 2)
✅ Tabla 7: salary_sort.csv — (7, 2)
✅ Tabla 8: contracts_by_type.csv — (32, 5)
✅ Tabla 9: segment_by_period.csv — (51, 3)


## Criterio de precios para revenue estimado (proxy)

En ausencia de datos de margen neto o facturación real, se asignan **valores unitarios fijos por producto** para construir un indicador de **revenue estimado**.

Este indicador se utiliza únicamente como **proxy analítico** para comparar la aportación relativa de las distintas categorías de producto y su evolución temporal dentro del dashboard.

No debe interpretarse como **facturación real**, **margen contable** ni ingreso financiero efectivo de easyMoney.

Los productos se agrupan en 5 familias (Tabla 13) para la página de Productos del dashboard.

| Familia | Productos | Precio unitario |
|---|---|---|
| Cuenta | em_acount, emc_account, payroll_account, payroll, em_account_p | €10 |
| Tarjetas | debit_card (€10), credit_card (€40) | €10 / €40 |
| Plan pensiones | pension_plan | €40 |
| Inversión | short_term_deposit, long_term_deposit, funds, securities | €40 |
| Financiación | loans, mortgage | €60 |

> **Revenue estimado total Mayo 2019 (proxy): ~€5,328,470** *(datos limpios, sin anomalías)*
> **Revenue estimado por cliente (proxy): ~€12.05**
>
> ⚠️ **Nota:** Los valores de revenue se calculan sobre `df_clean` (sin anomalías) para evitar distorsiones.
> La tabla `client_profile` sí incluye todos los clientes (con flags) para consistencia con los KPIs de volumen.

In [5]:
# ── Tablas 10-12: Revenue estimado (proxy) ────────────────────────────────
# Nota: last_partition y df_clean definidos en celdas anteriores

# Tabla 10: Revenue estimado (proxy) por período — universo completo limpio
revenue_period = []
for partition, group in df_clean.groupby('pk_partition'):
    total_rev = sum(group[col].sum() * price for col, price in price_map.items())
    revenue_period.append({
        'pk_partition': str(partition)[:10],
        'revenue_estimado_proxy': round(total_rev, 2),
        'total_clientes': len(group),
        'revenue_estimado_proxy_por_cliente': round(total_rev / len(group), 2) if len(group) > 0 else 0
    })

df_revenue = pd.DataFrame(revenue_period)
df_revenue.to_csv(OUTPUT_PATH + 'revenue_by_period.csv', index=False)
print(f"✅ Tabla 10: revenue_by_period.csv — {df_revenue.shape}")

# Tabla 11: Revenue estimado (proxy) por producto — última partición
# Nota: df_last definido en celda Tabla 5 (reutilizado aquí con .copy() para seguridad)
df_last = df_clean[df_clean['pk_partition'] == last_partition].copy()

revenue_product = pd.DataFrame({
    'producto':             list(price_map.keys()),
    'clientes_con_producto': [int(df_last[col].sum()) for col in price_map.keys()],
    'precio_unitario':      list(price_map.values())
})
revenue_product['revenue_estimado_proxy'] = (
    revenue_product['clientes_con_producto'] * revenue_product['precio_unitario']
).round(2)
revenue_product = revenue_product.sort_values('revenue_estimado_proxy', ascending=False)
revenue_product.to_csv(OUTPUT_PATH + 'revenue_by_product.csv', index=False)
print(f"✅ Tabla 11: revenue_by_product.csv — {revenue_product.shape}")

# Tabla 12: Revenue estimado (proxy) por región — última partición
revenue_region = df_last[['pk_cid', 'region_code'] + list(price_map.keys())].copy()
revenue_region['revenue_proxy'] = sum(
    revenue_region[col] * price for col, price in price_map.items()
)
revenue_region = (
    revenue_region.groupby('region_code')
    .agg(
        clientes              = ('pk_cid', 'count'),
        revenue_estimado_proxy = ('revenue_proxy', 'sum')
    )
    .reset_index()
    .sort_values('revenue_estimado_proxy', ascending=False)
)
revenue_region['revenue_estimado_proxy'] = revenue_region['revenue_estimado_proxy'].round(2)
revenue_region.to_csv(OUTPUT_PATH + 'revenue_by_region.csv', index=False)
print(f"✅ Tabla 12: revenue_by_region.csv — {revenue_region.shape}")

✅ Tabla 10: revenue_by_period.csv — (17, 4)
✅ Tabla 11: revenue_by_product.csv — (14, 4)
✅ Tabla 12: revenue_by_region.csv — (53, 3)


In [6]:
# ── Tablas 13-15: Familia de producto ─────────────────────────────────────
# Nota: df_last definido en celda anterior (Tablas 10-12)

familia_map = {
    'em_acount':          'Cuenta',
    'em_account_p':       'Cuenta',
    'emc_account':        'Cuenta',
    'payroll_account':    'Cuenta',
    'payroll':            'Cuenta',
    'debit_card':         'Tarjetas',
    'credit_card':        'Tarjetas',
    'pension_plan':       'Plan pensiones',
    'funds':              'Inversión',
    'securities':         'Inversión',
    'long_term_deposit':  'Inversión',
    'short_term_deposit': 'Inversión',
    'loans':              'Financiación',
    'mortgage':           'Financiación',
}

# ── Tabla 13: product_family_map.csv — tabla lookup (conectar en Power BI) ──
family_lookup = pd.DataFrame([
    {'nombre_tecnico': col,
     'familia':        fam,
     'producto':       label_map.get(col, col)}
    for col, fam in familia_map.items()
]).sort_values(['familia', 'producto'])

family_lookup.to_csv(OUTPUT_PATH + 'product_family_map.csv', index=False)
print(f"✅ Tabla 13: product_family_map.csv — {family_lookup.shape}")

# ── Tabla 14: revenue_by_family.csv — revenue por familia (mayo 2019) ──────
rows = []
for col, fam in familia_map.items():
    price    = price_map.get(col, 10)
    clientes = int(df_last[col].sum())
    rows.append({
        'familia':                fam,
        'nombre_tecnico':         col,
        'producto':               label_map.get(col, col),
        'precio_unitario':        price,
        'clientes_con_producto':  clientes,
        'revenue_estimado_proxy': clientes * price   # nombre consistente con otras tablas
    })

rev_detail = pd.DataFrame(rows)

rev_by_family = (
    rev_detail
    .groupby('familia')
    .agg(
        clientes_totales          = ('clientes_con_producto',  'sum'),
        revenue_estimado_proxy    = ('revenue_estimado_proxy', 'sum'),
        num_productos             = ('nombre_tecnico',         'count')
    )
    .reset_index()
    .sort_values('revenue_estimado_proxy', ascending=False)
)
rev_by_family['revenue_estimado_proxy'] = rev_by_family['revenue_estimado_proxy'].round(2)
rev_by_family.to_csv(OUTPUT_PATH + 'revenue_by_family.csv', index=False)
print(f"✅ Tabla 14: revenue_by_family.csv — {rev_by_family.shape}")
print(rev_by_family.to_string(index=False))

# ── Tabla 15: penetration_by_family.csv ────────────────────────────────────
pen_rows = []
for familia, cols_in_family in rev_detail.groupby('familia')['nombre_tecnico'].apply(list).items():
    mask            = df_last[cols_in_family].any(axis=1)
    clientes_unicos = int(mask.sum())
    pen_rows.append({
        'familia':         familia,
        'clientes_unicos': clientes_unicos,
        'penetracion_pct': round(clientes_unicos / len(df_last) * 100, 2)
    })

pen_by_family = (
    pd.DataFrame(pen_rows)
    .sort_values('penetracion_pct', ascending=False)
)
pen_by_family.to_csv(OUTPUT_PATH + 'penetration_by_family.csv', index=False)
print(f"\n✅ Tabla 15: penetration_by_family.csv — {pen_by_family.shape}")
print(pen_by_family.to_string(index=False))

✅ Tabla 13: product_family_map.csv — (14, 3)
✅ Tabla 14: revenue_by_family.csv — (5, 4)
       familia  clientes_totales  revenue_estimado_proxy  num_productos
        Cuenta            363593                 3635930              5
Plan pensiones             17306                  692240              1
      Tarjetas             47985                  623550              2
     Inversión              9215                  368600              4
  Financiación                53                    3180              2

✅ Tabla 15: penetration_by_family.csv — (5, 3)
       familia  clientes_unicos  penetracion_pct
        Cuenta           328401            74.34
      Tarjetas            44792            10.14
Plan pensiones            17306             3.92
     Inversión             8593             1.95
  Financiación               53             0.01


In [7]:
# ── Tabla 16: region_code → nombre de provincia ────────────────────────────

region_names = {
    '1': 'Álava', '2': 'Albacete', '3': 'Alicante', '4': 'Almería',
    '5': 'Ávila', '6': 'Badajoz', '7': 'Baleares', '8': 'Barcelona',
    '9': 'Burgos', '10': 'Cáceres', '11': 'Cádiz', '12': 'Castellón',
    '13': 'Ciudad Real', '14': 'Córdoba', '15': 'Coruña, A', '16': 'Cuenca',
    '17': 'Girona', '18': 'Granada', '19': 'Guadalajara', '20': 'Gipuzkoa',
    '21': 'Huelva', '22': 'Huesca', '23': 'Jaén', '24': 'León',
    '25': 'Lleida', '26': 'Rioja, La', '27': 'Lugo', '28': 'Madrid',
    '29': 'Málaga', '30': 'Murcia', '31': 'Navarra', '32': 'Ourense',
    '33': 'Asturias', '34': 'Palencia', '35': 'Palmas, Las', '36': 'Pontevedra',
    '37': 'Salamanca', '38': 'Tenerife', '39': 'Cantabria', '40': 'Segovia',
    '41': 'Sevilla', '42': 'Soria', '43': 'Tarragona', '44': 'Teruel',
    '45': 'Toledo', '46': 'Valencia', '47': 'Valladolid', '48': 'Bizkaia',
    '49': 'Zamora', '50': 'Zaragoza', '51': 'Ceuta', '52': 'Melilla',
    'UNKNOWN': 'UNKNOWN'
}

region_map = pd.DataFrame([
    {'region_code': code, 'provincia': name}
    for code, name in region_names.items()
])
region_map.to_csv(OUTPUT_PATH + 'region_map.csv', index=False)
print(f"✅ Tabla 16: region_map.csv — {region_map.shape}")

# ── Resumen final de exportación ───────────────────────────────────────────
print("\n" + "─" * 60)
print(f"✅ Exportación completa: 22 CSVs listos para Power BI")
print("─" * 60)
for i, fname in enumerate(sorted(os.listdir(OUTPUT_PATH)), 1):
    size_kb = os.path.getsize(OUTPUT_PATH + fname) / 1024
    print(f"  {i:>2}. {fname:<40} {size_kb:>8.1f} KB")
print("─" * 60)

✅ Tabla 16: region_map.csv — (53, 2)

────────────────────────────────────────────────────────────
✅ Exportación completa: 22 CSVs listos para Power BI
────────────────────────────────────────────────────────────
   1. age_sort.csv                                  0.1 KB
   2. client_profile.csv                        52921.4 KB
   3. contracts_by_type.csv                         1.1 KB
   4. contracts_by_type_v2.csv                      1.1 KB
   5. crosssell_funnel.csv                          1.2 KB
   6. kpi_rentabilidad.csv                          0.7 KB
   7. kpi_summary.csv                               1.8 KB
   8. penetration_by_family.csv                     0.2 KB
   9. period_summary.csv                            1.4 KB
  10. product_family_map.csv                        0.5 KB
  11. product_penetration.csv                       1.5 KB
  12. product_penetration_long.csv                  0.5 KB
  13. region_map.csv                                0.7 KB
  14. retention_by_p

## Respuestas a Preguntas de Negocio — Comité Ejecutivo

Responde explícitamente las dos preguntas clave de Carol (CEO).

- **Pregunta 1:** ¿Cuántos productos se vendieron en el último período?
- **Pregunta 2:** ¿Quién contrata más, clientes nuevos o existentes?

In [8]:
# ── PREGUNTA 1: ¿Cuántos productos se vendieron en Mayo 2019? ─────────────

# Reload si se ejecuta de forma independiente (sin ejecutar celda Setup)
try:
    _ = df.shape
    _ = df_clean.shape
    _ = OUTPUT_PATH
    _ = product_cols
    _ = last_partition
    _ = anomaly_flags
except NameError:
    BASE = 'C:\\Users\\farno\\OneDrive\\Desktop\\Data science & AI - Nuclio School\\proyecto final TFM\\tfm-fintech-easymoney\\data\\processed\\'
    OUTPUT_PATH = BASE + 'powerbi\\'
    product_cols = [
        'short_term_deposit','loans','mortgage','funds','securities',
        'long_term_deposit','credit_card','payroll','pension_plan',
        'payroll_account','emc_account','debit_card','em_account_p','em_acount'
    ]
    anomaly_flags = ['age_anomaly', 'deceased_anomaly', 
                'entry_date_anomaly', 'salary_anomaly']
    df = pd.read_parquet(BASE + 'master_df_flags.parquet')
    df_clean = df[~df[anomaly_flags].any(axis=1)]
    last_partition = df['pk_partition'].max()

# ── Filtro de última partición (clientes limpios) ─────────────────────────
last = df_clean[df_clean['pk_partition'] == last_partition].copy()
last_str = str(last_partition)[:10]

# ── Contratos de clientes existentes: new_contracts = delta vs período anterior
contracts_exist = int(last[last['is_new_client'] == 0]['new_contracts'].sum())

# ── Contratos de clientes nuevos: total_products (todos son nuevos en su primer período)
# Equivalente a fillna(0) en el shift de 01-eda: si no había período anterior → 0 previo
mask_new        = last['is_new_client'] == 1
contracts_new   = int(last[mask_new]['new_contracts'].sum())
n_new_clients   = int(mask_new.sum())
n_exist_clients = int((~mask_new).sum())

total_contracts = contracts_exist + contracts_new

print("=" * 65)
print(f"  PREGUNTA 1: ¿Cuántos productos se vendieron en {last_str}?")
print("=" * 65)
print(f"\n  Clientes existentes ({n_exist_clients:>7,}):  {contracts_exist:>7,} contratos nuevos")
print(f"  Clientes nuevos     ({n_new_clients:>7,}):  {contracts_new:>7,} contratos")
print(f"\n  {'─' * 55}")
print(f"  TOTAL productos contratados {last_str}:   {total_contracts:>7,}")
print(f"\n  Métricas adicionales:")
print(f"  · Ratio contratos/cliente existente:  {contracts_exist/n_exist_clients:.4f}")
print(f"  · Productos por cliente nuevo al alta: {contracts_new/n_new_clients:.2f}" if n_new_clients > 0 else "")

  PREGUNTA 1: ¿Cuántos productos se vendieron en 2019-05-28?

  Clientes existentes (437,670):   16,975 contratos nuevos
  Clientes nuevos     (  4,082):    1,690 contratos

  ───────────────────────────────────────────────────────
  TOTAL productos contratados 2019-05-28:    18,665

  Métricas adicionales:
  · Ratio contratos/cliente existente:  0.0388
  · Productos por cliente nuevo al alta: 0.41


In [9]:
# ── PREGUNTA 2: ¿Quién compra — clientes nuevos o existentes? ────────────
# Análisis sobre toda la serie temporal (17 períodos) + corrección del bug

# ── Fix vectorizado: contratos reales por tipo de cliente ─────────────────
df_no_first = df_clean[df_clean['pk_partition'] != df_clean['pk_partition'].min()].copy()

is_new = df_no_first['is_new_client'].astype(int)
df_no_first['contracts_corr'] = (
    is_new * df_no_first[product_cols].sum(axis=1) +           # nuevo: total productos al alta
    (1 - is_new) * df_no_first['new_contracts']                # existente: delta vs período anterior
)

# ── Tabla corregida: contratos por tipo y período ─────────────────────────
contracts_v2 = (
    df_no_first
    .groupby(['pk_partition', 'is_new_client'])
    .agg(
        contracts    = ('contracts_corr', 'sum'),
        client_count = ('pk_cid', 'count')
    )
    .reset_index()
)
contracts_v2['pk_partition'] = contracts_v2['pk_partition'].astype(str).str[:10]
contracts_v2['tipo']         = contracts_v2['is_new_client'].map({0: 'Existente', 1: 'Nuevo'})

# Guardar versión corregida (no sobreescribe el original)
contracts_v2.to_csv(OUTPUT_PATH + 'contracts_by_type_v2.csv', index=False)
print(f"✓ contracts_by_type_v2.csv guardado")

# ── Resumen Mayo 2019 ─────────────────────────────────────────────────────
may_v2   = contracts_v2[contracts_v2['pk_partition'] == str(last_partition)[:10]]
c_exist  = int(may_v2[may_v2['tipo'] == 'Existente']['contracts'].values[0])
c_new    = int(may_v2[may_v2['tipo'] == 'Nuevo']['contracts'].values[0])
c_total  = c_exist + c_new

print("\n" + "=" * 65)
print(f"  PREGUNTA 2: ¿Quién compra — nuevos o existentes? (Mayo 2019)")
print("=" * 65)
print(f"\n  Clientes EXISTENTES → {c_exist:>7,} contratos  ({c_exist/c_total*100:.1f}%)")
print(f"  Clientes NUEVOS     → {c_new:>7,} contratos  ({c_new/c_total*100:.1f}%)")
print(f"  {'─' * 45}")
print(f"  TOTAL                 {c_total:>7,} contratos")

# ── Tendencia histórica ────────────────────────────────────────────────────
trend = (
    contracts_v2
    .groupby(['pk_partition', 'tipo'])['contracts']
    .sum()
    .unstack('tipo')
    .fillna(0)
    .astype(int)
)
trend['Total']      = trend['Existente'] + trend['Nuevo']
trend['%_Existente'] = (trend['Existente'] / trend['Total'] * 100).round(1)
trend['%_Nuevo']     = (trend['Nuevo']     / trend['Total'] * 100).round(1)

print("\n── Tendencia histórica completa ─────────────────────────────────")
print(f"{'Período':<14} {'Existente':>10} {'Nuevo':>8} {'Total':>8} {'%_Exist':>9} {'%_Nuevo':>8}")
print("─" * 62)
for period, row in trend.iterrows():
    print(f"{period:<14} {int(row['Existente']):>10,} {int(row['Nuevo']):>8,} "
          f"{int(row['Total']):>8,} {row['%_Existente']:>8.1f}% {row['%_Nuevo']:>7.1f}%")

print("\n── Lectura de negocio ──────────────────────────────────────────")
avg_exist_pct = trend['%_Existente'].mean()
avg_new_pct   = trend['%_Nuevo'].mean()
print(f"  · Promedio histórico: {avg_exist_pct:.0f}% clientes existentes / {avg_new_pct:.0f}% nuevos")
print(f"  · La estrategia de penetración está funcionando en el portafolio")
print(f"    actual: {avg_exist_pct:.0f} de cada 100 contratos vienen de alguien que ya era cliente")
print(f"  · El mix cambia en Jul-Oct 2018 (pico de nuevos clientes)")
_peak_idx = trend['Nuevo'].idxmax()
_peak_new = int(trend.loc[_peak_idx, 'Nuevo'])
print(f"    coincidiendo con la incorporación masiva de {_peak_new:,} clientes en {_peak_idx}")

✓ contracts_by_type_v2.csv guardado

  PREGUNTA 2: ¿Quién compra — nuevos o existentes? (Mayo 2019)

  Clientes EXISTENTES →  16,975 contratos  (90.9%)
  Clientes NUEVOS     →   1,690 contratos  (9.1%)
  ─────────────────────────────────────────────
  TOTAL                  18,665 contratos

── Tendencia histórica completa ─────────────────────────────────
Período         Existente    Nuevo    Total   %_Exist  %_Nuevo
──────────────────────────────────────────────────────────────
2018-02-28         11,139    4,015   15,154     73.5%    26.5%
2018-03-28         11,578    3,637   15,215     76.1%    23.9%
2018-04-28         10,567    3,106   13,673     77.3%    22.7%
2018-05-28         10,430    3,170   13,600     76.7%    23.3%
2018-06-28         14,503    3,013   17,516     82.8%    17.2%
2018-07-28         13,236   12,653   25,889     51.1%    48.9%
2018-08-28         14,378   11,081   25,459     56.5%    43.5%
2018-09-28         15,364   15,913   31,277     49.1%    50.9%
2018-10-28 

## Definición Formal de KPIs — Tarea 1

Todos los KPIs se calculan sobre `df_clean` (sin anomalías) excepto los de volumen que usan el universo completo.

| KPI | Fórmula | Granularidad | Fuente CSV |
|-----|---------|-------------|--------|
| **total_clients** | `COUNT(pk_cid)` | Mensual | `period_summary.csv` |
| **active_clients** | `SUM(active_customer == 1)` | Mensual | `period_summary.csv` |
| **pct_active** | `active_clients / total_clients × 100` | Mensual | calculado |
| **new_clients** | `SUM(is_new_client == 1)` | Mensual | `period_summary.csv` |
| **new_contracts** | `SUM(new_contracts)` exist. + `SUM(total_products)` nuevos | Mensual | `contracts_by_type_v2.csv` |
| **avg_products** | `MEAN(total_products)` | Mensual | `period_summary.csv` |
| **pct_0_products** | `COUNT(total_products==0) / total_clients × 100` | Mensual | `kpi_summary.csv` |
| **pct_crosssell** | `COUNT(total_products>=2) / total_clients × 100` | Mensual | `kpi_summary.csv` |
| **penetracion_producto** | `MEAN(col_producto) × 100` | Mensual × Producto | `product_penetration.csv` |
| **revenue_proxy** | `SUM(producto × precio_unitario)` | Mensual / Producto / Región | `revenue_by_period.csv` |

> **Revenue es proxy analítico** — no representa facturación real ni margen contable.

---

## KPIs de Rentabilidad y Venta Cruzada

Esta sección responde al objetivo estratégico de EasyMoney: **pivotar hacia la rentabilidad y la venta cruzada**.

Los KPIs de volumen anteriores son necesarios pero no suficientes para medir ese pivote. Aquí se calculan y exportan los indicadores que lo cuantifican:

| Tabla | Archivo | Contenido |
|---|---|---|
| 18 | `revenue_by_segment.csv` | Revenue estimado por segmento (TOP / PARTICULARES / UNIVERSITARIO) — mayo 2019 |
| 19 | `kpi_rentabilidad.csv` | ARPU proxy (revenue estimado / cliente activo) — serie temporal 17 períodos |
| 20 | `retention_by_period.csv` | Tasa de retención y churn mes a mes |
| 21 | `crosssell_funnel.csv` | Evolución del mix de clientes por nº de productos (funnel 0→1→2+) |
| 22 | `revenue_uplift.csv` | Revenue medio por nº de productos contratados — cuantifica el valor del cross-sell |

In [10]:
# ── Tabla 18: Revenue por segmento — Mayo 2019 ────────────────────────────────
seg_revenue = []
for seg, group in df_clean[df_clean['pk_partition'] == last_partition].groupby('segment'):
    total_rev  = sum(group[col].sum() * price for col, price in price_map.items())
    n_clientes = len(group)
    n_activos  = int((group['active_customer'] == 1.0).sum())
    seg_revenue.append({
        'segmento':                   seg,
        'clientes_totales':           n_clientes,
        'clientes_activos':           n_activos,
        'revenue_estimado_proxy':     round(total_rev, 2),
        'revenue_proxy_por_cliente':  round(total_rev / n_clientes, 2),
        'revenue_proxy_por_activo':   round(total_rev / n_activos, 2) if n_activos > 0 else 0,
        'avg_productos':              round(group['total_products'].mean(), 2),
    })

df_seg_rev = pd.DataFrame(seg_revenue).sort_values('revenue_estimado_proxy', ascending=False)
df_seg_rev.to_csv(OUTPUT_PATH + 'revenue_by_segment.csv', index=False)
print(f"✅ Tabla 18: revenue_by_segment.csv — {df_seg_rev.shape}")
print(df_seg_rev.to_string(index=False))

# ── Tabla 19: ARPU proxy por período (serie temporal) ─────────────────────────
arpu_period = []
for part, group in df_clean.groupby('pk_partition'):
    total_rev = sum(group[col].sum() * price for col, price in price_map.items())
    n_activos = int((group['active_customer'] == 1.0).sum())
    n_total   = len(group)
    arpu_period.append({
        'pk_partition':               str(part)[:10],
        'revenue_estimado_proxy':     round(total_rev, 2),
        'clientes_activos':           n_activos,
        'arpu_proxy':                 round(total_rev / n_activos, 2) if n_activos > 0 else 0,
        'revenue_proxy_por_cliente':  round(total_rev / n_total, 2),
    })

df_arpu = pd.DataFrame(arpu_period)
df_arpu.to_csv(OUTPUT_PATH + 'kpi_rentabilidad.csv', index=False)
print(f"\n✅ Tabla 19: kpi_rentabilidad.csv — {df_arpu.shape}")
print(f"\nARPU proxy por período:")
print(df_arpu[['pk_partition', 'clientes_activos', 'revenue_estimado_proxy', 'arpu_proxy']].to_string(index=False))
print(f"\n  ARPU actual (mayo 2019): €{df_arpu.iloc[-1]['arpu_proxy']:.2f} / cliente activo")

✅ Tabla 18: revenue_by_segment.csv — (3, 7)
          segmento  clientes_totales  clientes_activos  revenue_estimado_proxy  revenue_proxy_por_cliente  revenue_proxy_por_activo  avg_productos
03 - UNIVERSITARIO            288146             79218                 2693780                       9.35                     34.00           0.87
 02 - PARTICULARES            146443             84911                 2339680                      15.98                     27.55           1.18
          01 - TOP              7163              6933                  290040                      40.49                     41.83           2.10

✅ Tabla 19: kpi_rentabilidad.csv — (17, 5)

ARPU proxy por período:
pk_partition  clientes_activos  revenue_estimado_proxy  arpu_proxy
  2018-01-28            107855                 3550880       32.92
  2018-02-28            110744                 3647840       32.94
  2018-03-28            113402                 3743370       33.01
  2018-04-28            115904 

In [11]:
# ── ALERTA: Tendencia ARPU — KPI crítico para el Comité ───────────────────
arpu_inicio = df_arpu.iloc[0]['arpu_proxy']
arpu_fin    = df_arpu.iloc[-1]['arpu_proxy']
arpu_delta  = arpu_fin - arpu_inicio
arpu_delta_pct = (arpu_delta / arpu_inicio) * 100

arpu_max = df_arpu['arpu_proxy'].max()
arpu_max_period = df_arpu.loc[df_arpu['arpu_proxy'].idxmax(), 'pk_partition']

print("=" * 60)
print("  ⚠️  ALERTA — ARPU proxy en tendencia DECRECIENTE")
print("=" * 60)
print(f"\n  ARPU enero 2018:   €{arpu_inicio:.2f} / cliente activo")
print(f"  ARPU mayo 2019:    €{arpu_fin:.2f} / cliente activo")
print(f"  Variación:         {arpu_delta:+.2f} EUR  ({arpu_delta_pct:+.1f}%)")
print(f"\n  Pico histórico:    €{arpu_max:.2f}  ({arpu_max_period})")
print(f"\n  Causa probable:")
print(f"  → La entrada masiva de clientes universitarios (jul-oct 2018)")
print(f"    incorporó ~84k clientes con 0 productos → diluyeron el ARPU medio.")
print(f"  → El cross-sell no ha compensado ese efecto: pct_crosssell actual")
print(f"    es 14.3%, lejos del objetivo del 20%.")
print(f"\n  Impacto potencial si pct_crosssell alcanza 20%:")
n_activos_may = int(df_arpu.iloc[-1]['clientes_activos'])
rev_actual    = df_arpu.iloc[-1]['revenue_estimado_proxy']
# Estimacion: 5.7pp adicionales de crosssell ~ clientes 1-prod que pasan a 2 prod
n_adicionales_crosssell = int(n_activos_may * 0.057)
rev_adicional = n_adicionales_crosssell * 10  # incremento minimo: +1 producto a precio 10
print(f"    ~{n_adicionales_crosssell:,} clientes pasarian de 1 a 2+ productos")
print(f"    Revenue adicional estimado (proxy, minimo): +€{rev_adicional:,}/mes")
print(f"\n  Recomendacion para el Comité:")
print(f"  Incluir ARPU como KPI de seguimiento mensual — objetivo: recuperar €32+ en 12 meses.")

  ⚠️  ALERTA — ARPU proxy en tendencia DECRECIENTE

  ARPU enero 2018:   €32.92 / cliente activo
  ARPU mayo 2019:    €31.12 / cliente activo
  Variación:         -1.80 EUR  (-5.5%)

  Pico histórico:    €33.01  (2018-03-28)

  Causa probable:
  → La entrada masiva de clientes universitarios (jul-oct 2018)
    incorporó ~84k clientes con 0 productos → diluyeron el ARPU medio.
  → El cross-sell no ha compensado ese efecto: pct_crosssell actual
    es 14.3%, lejos del objetivo del 20%.

  Impacto potencial si pct_crosssell alcanza 20%:
    ~9,750 clientes pasarian de 1 a 2+ productos
    Revenue adicional estimado (proxy, minimo): +€97,500/mes

  Recomendacion para el Comité:
  Incluir ARPU como KPI de seguimiento mensual — objetivo: recuperar €32+ en 12 meses.


In [12]:
# ── Tabla 20: Retention / Churn proxy por período ─────────────────────────────
# Para cada período t, contamos cuántos clientes de t-1 siguen presentes en t.
# Retention rate = clientes_retenidos / clientes_prev
# Churn rate     = 1 - retention_rate
# Nota: solo detecta clientes que desaparecen del registro, no cancelaciones reales.

periods_sorted = sorted(df_clean['pk_partition'].unique())
retention_rows = []

for i in range(1, len(periods_sorted)):
    prev = periods_sorted[i - 1]
    curr = periods_sorted[i]
    clients_prev = set(df_clean[df_clean['pk_partition'] == prev]['pk_cid'])
    clients_curr = set(df_clean[df_clean['pk_partition'] == curr]['pk_cid'])

    retained   = len(clients_prev & clients_curr)
    ret_rate   = round(retained / len(clients_prev) * 100, 2) if clients_prev else 0

    retention_rows.append({
        'pk_partition':       str(curr)[:10],
        'prev_partition':     str(prev)[:10],
        'clientes_prev':      len(clients_prev),
        'clientes_curr':      len(clients_curr),
        'clientes_retenidos': retained,
        'clientes_perdidos':  len(clients_prev - clients_curr),
        'clientes_nuevos':    len(clients_curr - clients_prev),
        'retention_rate_pct': ret_rate,
        'churn_rate_pct':     round(100 - ret_rate, 2),
    })

df_retention = pd.DataFrame(retention_rows)
df_retention.to_csv(OUTPUT_PATH + 'retention_by_period.csv', index=False)

print("✅ Tabla 20: retention_by_period.csv")
print(f"\n{'Período':<14} {'Prev':>8} {'Curr':>8} {'Retenidos':>10} {'Retention%':>11} {'Churn%':>8}")
print("─" * 65)
for _, row in df_retention.iterrows():
    print(f"{row['pk_partition']:<14} {int(row['clientes_prev']):>8,} {int(row['clientes_curr']):>8,} "
          f"{int(row['clientes_retenidos']):>10,} {row['retention_rate_pct']:>10.1f}% {row['churn_rate_pct']:>7.1f}%")

avg_ret = df_retention['retention_rate_pct'].mean()
print(f"\n  Media histórica → Retention: {avg_ret:.1f}%  |  Churn: {100 - avg_ret:.1f}%")
print(f"  ⚠️  Churn aquí = clientes que no aparecen en la siguiente partición.")
print(f"      No equivale a cancelación real — puede incluir depuración del sistema.")

✅ Tabla 20: retention_by_period.csv

Período            Prev     Curr  Retenidos  Retention%   Churn%
─────────────────────────────────────────────────────────────────
2018-02-28      238,991  242,045    238,249       99.7%     0.3%
2018-03-28      242,045  244,720    241,301       99.7%     0.3%
2018-04-28      244,720  246,914    243,832       99.6%     0.4%
2018-05-28      246,914  249,404    246,164       99.7%     0.3%
2018-06-28      249,404  251,533    248,585       99.7%     0.3%
2018-07-28      251,533  335,669    251,532      100.0%     0.0%
2018-08-28      335,669  351,727    334,684       99.7%     0.3%
2018-09-28      351,727  372,285    350,775       99.7%     0.3%
2018-10-28      372,285  399,645    371,037       99.7%     0.3%
2018-11-28      399,645  414,680    398,170       99.6%     0.4%
2018-12-28      414,680  420,982    413,343       99.7%     0.3%
2019-01-28      420,982  425,927    418,435       99.4%     0.6%
2019-02-28      425,927  430,949    424,511       99

In [13]:
# ── Tabla 21: Funnel de Cross-sell por período ────────────────────────────────
# Evolución del mix de clientes según nº de productos: 0 / 1 / 2 / 3+
# KPI principal: pct_crosssell = % clientes con 2+ productos

funnel_rows = []
for part, group in df_clean.groupby('pk_partition'):
    n_total = len(group)
    n_0     = int((group['total_products'] == 0).sum())
    n_1     = int((group['total_products'] == 1).sum())
    n_2     = int((group['total_products'] == 2).sum())
    n_3plus = int((group['total_products'] >= 3).sum())

    funnel_rows.append({
        'pk_partition':      str(part)[:10],
        'total_clientes':    n_total,
        'n_0_productos':     n_0,
        'n_1_producto':      n_1,
        'n_2_productos':     n_2,
        'n_3plus_productos': n_3plus,
        'pct_0_productos':   round(n_0 / n_total * 100, 2),
        'pct_1_producto':    round(n_1 / n_total * 100, 2),
        'pct_crosssell':     round((n_2 + n_3plus) / n_total * 100, 2),
        'pct_3plus':         round(n_3plus / n_total * 100, 2),
    })

df_funnel = pd.DataFrame(funnel_rows)
df_funnel.to_csv(OUTPUT_PATH + 'crosssell_funnel.csv', index=False)

print("✅ Tabla 21: crosssell_funnel.csv")
print(f"\n  Evolución del funnel de cross-sell (objetivo directiva: pct_crosssell > 20%)")
print(f"\n{'Período':<14} {'0 prod':>8} {'1 prod':>8} {'2+ prod':>8} {'3+ prod':>8}")
print("─" * 50)
for _, row in df_funnel.iterrows():
    print(f"{row['pk_partition']:<14} "
          f"{row['pct_0_productos']:>7.1f}% "
          f"{row['pct_1_producto']:>7.1f}% "
          f"{row['pct_crosssell']:>7.1f}% "
          f"{row['pct_3plus']:>7.1f}%")

delta = df_funnel['pct_crosssell'].iloc[-1] - df_funnel['pct_crosssell'].iloc[0]
print(f"\n  Variación pct_crosssell (ene 2018 → may 2019): {delta:+.1f} pp")
print(f"  Valor actual (mayo 2019): {df_funnel['pct_crosssell'].iloc[-1]:.1f}%  |  Objetivo: >20%")

✅ Tabla 21: crosssell_funnel.csv

  Evolución del funnel de cross-sell (objetivo directiva: pct_crosssell > 20%)

Período          0 prod   1 prod  2+ prod  3+ prod
──────────────────────────────────────────────────
2018-01-28         1.8%    82.6%    15.7%     5.6%
2018-02-28         1.8%    82.1%    16.1%     5.8%
2018-03-28         1.8%    81.5%    16.7%     6.1%
2018-04-28         1.8%    81.1%    17.1%     6.3%
2018-05-28         1.7%    81.1%    17.2%     6.2%
2018-06-28         1.8%    80.5%    17.7%     6.7%
2018-07-28        23.0%    63.3%    13.7%     5.4%
2018-08-28        22.8%    64.1%    13.1%     5.0%
2018-09-28        22.8%    64.3%    13.0%     4.9%
2018-10-28        23.7%    63.5%    12.8%     4.8%
2018-11-28        24.3%    63.0%    12.7%     4.8%
2018-12-28        24.6%    62.2%    13.2%     5.1%
2019-01-28        24.5%    62.6%    12.9%     4.6%
2019-02-28        24.8%    61.8%    13.4%     5.1%
2019-03-28        24.7%    61.5%    13.8%     5.2%
2019-04-28        2

In [14]:
# ── Tabla 22: Revenue Uplift por Cross-sell ───────────────────────────────────
# Cuantifica el valor económico de cada producto adicional contratado.
# Baseline = revenue medio de clientes con 1 producto.
# Cálculo sobre última partición (mayo 2019), datos limpios.

df_last = df_clean[df_clean['pk_partition'] == last_partition].copy()
df_last['revenue_proxy'] = sum(df_last[col] * price for col, price in price_map.items())

uplift_rows = []
for n_prod in range(0, 7):
    mask  = df_last['total_products'] == n_prod if n_prod < 6 else df_last['total_products'] >= 6
    label = str(n_prod) if n_prod < 6 else '6+'
    group = df_last[mask]
    if len(group) == 0:
        continue
    uplift_rows.append({
        'n_productos':         label,
        'n_clientes':          len(group),
        'pct_clientes':        round(len(group) / len(df_last) * 100, 2),
        'revenue_medio_proxy': round(group['revenue_proxy'].mean(), 2),
        'revenue_total_proxy': round(group['revenue_proxy'].sum(), 2),
    })

df_uplift = pd.DataFrame(uplift_rows)

baseline = df_uplift.loc[df_uplift['n_productos'] == '1', 'revenue_medio_proxy'].values
if len(baseline) > 0:
    b = baseline[0]
    df_uplift['uplift_vs_1prod_pct'] = df_uplift['revenue_medio_proxy'].apply(
        lambda x: round((x - b) / b * 100, 1) if b > 0 else 0
    )

df_uplift.to_csv(OUTPUT_PATH + 'revenue_uplift.csv', index=False)

print("✅ Tabla 22: revenue_uplift.csv")
print(f"\n  Revenue uplift por nº de productos — Mayo {str(last_partition)[:7]}")
print(f"  Baseline = clientes con 1 producto\n")
print(f"{'N Prod':>7} {'Clientes':>10} {'%Total':>7} {'Rev.Medio€':>12} {'Uplift vs 1P':>14}")
print("─" * 56)
for _, row in df_uplift.iterrows():
    uplift_str = f"{row['uplift_vs_1prod_pct']:+.0f}%" if 'uplift_vs_1prod_pct' in df_uplift.columns else "—"
    print(f"{row['n_productos']:>7} {int(row['n_clientes']):>10,} {row['pct_clientes']:>6.1f}% "
          f"{row['revenue_medio_proxy']:>12.2f} {uplift_str:>14}")

rev_2plus = df_uplift.loc[df_uplift['n_productos'].isin(['2','3','4','5','6+']), 'revenue_total_proxy'].sum()
n_2plus   = df_uplift.loc[df_uplift['n_productos'].isin(['2','3','4','5','6+']), 'n_clientes'].sum()
rev_1p    = baseline[0] if len(baseline) > 0 else 0
rev_at_risk = rev_2plus - rev_1p * n_2plus

print(f"\n  Revenue de clientes con 2+ productos:          €{rev_2plus:>10,.0f}")
print(f"  Revenue si tuvieran solo 1 producto (baseline): €{rev_1p * n_2plus:>10,.0f}")
print(f"  → Revenue generado por cross-sell:              €{rev_at_risk:>10,.0f}")

✅ Tabla 22: revenue_uplift.csv

  Revenue uplift por nº de productos — Mayo 2019-05
  Baseline = clientes con 1 producto

 N Prod   Clientes  %Total   Rev.Medio€   Uplift vs 1P
────────────────────────────────────────────────────────
      0    110,547   25.0%         0.00          -100%
      1    268,002   60.7%        10.17            +0%
      2     38,651    8.8%        23.62          +132%
      3     11,484    2.6%        52.77          +419%
      4      8,468    1.9%        71.81          +606%
      5      3,338    0.8%        88.77          +773%
     6+      1,262    0.3%        58.38          +474%

  Revenue de clientes con 2+ productos:          € 2,497,056
  Revenue si tuvieran solo 1 producto (baseline): €   642,775
  → Revenue generado por cross-sell:              € 1,854,281


---

## Definición Formal de KPIs — Rentabilidad y Cross-sell

| KPI | Definición | Fórmula | CSV |
|---|---|---|---|
| **Revenue por segmento** | Revenue estimado total y por cliente activo, desglosado por TOP / PARTICULARES / UNIVERSITARIO. Permite priorizar la inversión comercial por segmento | `Σ(producto × precio_unitario)` agrupado por segmento | `revenue_by_segment.csv` |
| **ARPU proxy** | Revenue estimado por cliente activo. Mide la rentabilidad media de los clientes que generan ingresos | `revenue_estimado_proxy / clientes_activos` | `kpi_rentabilidad.csv` |
| **Retention rate** | % de clientes del período t-1 que siguen activos en t. Proxy de fidelización | `clientes_retenidos / clientes_prev × 100` | `retention_by_period.csv` |
| **Churn rate** | % de clientes perdidos entre períodos consecutivos | `100 − retention_rate` | `retention_by_period.csv` |
| **pct_crosssell** | % de clientes con 2 o más productos contratados. KPI principal del pivote estratégico hacia la venta cruzada | `COUNT(total_products ≥ 2) / total_clientes × 100` | `crosssell_funnel.csv` |
| **Revenue uplift** | Revenue medio de un cliente con N productos respecto al baseline (1 producto). Cuantifica el valor económico de cada producto adicional vendido | `(rev_medio_Nprod − rev_medio_1prod) / rev_medio_1prod × 100` | `revenue_uplift.csv` |

> **Nota:** todos estos KPIs usan `df_clean` (sin anomalías) y precios unitarios proxy — no representan facturación real.
> El pivote estratégico se operacionaliza en tres KPIs clave: `pct_crosssell`, `ARPU proxy` y `revenue_uplift`.